<a href="https://colab.research.google.com/github/Nandish4470/Analyzer_2.0/blob/main/analyzer_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Google Colab Notebook - Elite Indian Railway Tender Parser v5.1 (Table-Based, Corrected)
#
# STEP 1: Run this cell to install all required libraries.
!pip install -q pdfplumber tabula-py PyPDF2 pandas tqdm openpyxl
print("✓ All libraries installed.")

# ----------------------------------------------
# STEP 2: Define all Parsing Functions
# (Run this cell to load functions into memory)
# ----------------------------------------------
import os
import re
import io
import json
import pickle
import math
import time
import traceback
from collections import defaultdict, Counter
from tqdm import tqdm
from datetime import datetime
from decimal import Decimal, InvalidOperation

import pandas as pd
import numpy as np
import pdfplumber
import PyPDF2
try:
    import tabula
except Exception as e:
    print(f"Tabula-py might have issues: {e}")
    tabula = None

from google.colab import files
from IPython.display import Markdown, display, HTML

# --- Helper Functions (v5.1) ---

def to_number(s):
    """Robustly converts a string to a float, handling commas, (negatives), and N/A values."""
    if s is None:
        return np.nan
    if isinstance(s, (int, float, Decimal, np.integer, np.floating)):
        try:
            return float(s)
        except:
            return np.nan

    st = str(s).strip()
    if not st or st.lower() in ("na", "n/a", "-", "nan", "none", "nil"):
        return np.nan

    st = st.replace("\xa0", "").replace("$", "").replace("€", "").replace("£", "").replace("₹", "")

    neg = False
    if st.startswith("(") and st.endswith(")"):
        neg = True
        st = st[1:-1].strip()

    st = st.replace(",", "")
    st_clean = re.sub(r"[^0-9eE\.\-]+", "", st)

    if st_clean in ("", ".", "-", "+"):
        return np.nan
    try:
        val = float(st_clean)
        return -val if neg else val
    except (ValueError, InvalidOperation):
        return np.nan

def normalize_text(text):
    """Cleans and normalizes text from PDF."""
    if not text:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[\s\u00a0\u200b\u200c\u200d\ufeff]+", " ", text)
    return text.strip()

def confidence_msg(task, method, score):
    """Formats a confidence log message."""
    emoji = "✅" if score >= 85 else "⚠️" if score >= 70 else "❌"
    return f"{emoji} {task} → {method} | {score}% confidence"

def parse_nit_header_hybrid(page_text, page_tables):
    """
    Robust NIT Header parser (v5.1).
    Uses a table-first approach and falls back to Regex.
    """
    nit = {}

    # --- Table-based parsing (Priority) ---
    try:
        if page_tables:
            for table in page_tables:
                if not table or len(table) < 2:
                    continue

                # Check for 2-column key-value format (common in some tenders)
                if len(table[0]) == 2:
                    for row in table:
                        k = normalize_text(row[0])
                        v = normalize_text(row[1])
                        if k and v: nit[k] = v

                # Check for 4 or 6 column key-value format (common in NIT-BCT tenders)
                elif len(table[0]) in (4, 6):
                    for row in table:
                        for i in range(0, len(row), 2):
                            k = normalize_text(row[i])
                            v = normalize_text(row[i+1])
                            if k and v: nit[k] = v

    except Exception as e:
        pass # Fallback to regex

    # --- Regex-based parsing (Fallback) ---
    header_text_multiline = page_text.replace('\r','\n')

    patterns = {
        'Tender No': r'Tender No\s*[:\s]*\s*([A-Za-z0-9\-\_\/]+)',
        'Name of Work': r'Name of Work\s*[:\s]*\s*([\s\S]*?)(?=Bidding type)',
        'Bidding type': r'Bidding type\s*[:\s]*\s*([^\n]+)',
        'Tender Type': r'Tender Type\s*[:\s]*\s*([^\n]+)',
        'Bidding System': r'Bidding System\s*[:\s]*\s*([^\n]+)',
        'Tender Closing Date Time': r'Tender Closing Date\s+Time\s*[:\s]*\s*([^\n]+)',
        'Advertised Value': r'Advertised Value\s*[:\s]*\s*([\d\.,]+)',
        'Earnest Money (Rs.)': r'Earnest Money \(Rs\.\)\s*[:\s]*\s*([\d\.,]+)',
        'Period of Completion': r'Period of Completion\s*[:\s]*\s*([^\n]+)',
        'Contract Type': r'Contract Type\s*[:\s]*\s*([^\n]+)',
        'Are JV allowed to bid': r'Are JV allowed to bid\s*[:\s]*\s*([^\n]+)',
        'Bidding Start Date': r'Bidding Start Date\s*[:\s]*\s*([^\n]+)',
    }

    for k, pat in patterns.items():
        if k not in nit:
            m = re.search(pat, header_text_multiline, re.IGNORECASE)
            if m:
                nit[k] = normalize_text(m.group(1))

    # --- Clean known messy fields ---
    if 'Name of Work' in nit:
        nit['Name of Work'] = nit['Name of Work'].split('\n')[0].strip()

    # Final check for critical fields
    if not nit.get('Advertised Value'):
        m = re.search(r'Advertised Value\s*([\d\.,]+)', header_text_multiline)
        if m: nit['Advertised Value'] = normalize_text(m.group(1))

    if not nit.get('Earnest Money (Rs.)') or to_number(nit.get('Earnest Money (Rs.)')) == 0:
        m = re.search(r'Earnest Money \(Rs\.\)\s*[:\s]*\s*([\d\.,]+)', header_text_multiline)
        if m: nit['Earnest Money (Rs.)'] = normalize_text(m.group(1))

    return nit

def parse_schedule_header(line):
    """
    Identifies schedule headers from a line of text.
    e.g., 'Item-1 Schedule A1 (2.0 Earthwork)'
    """
    line = normalize_text(str(line))

    # Pattern: Item-1 Schedule A1 (2.0 Earthwork)
    m = re.search(r'Item-(\d+)\s+(Schedule\s+[A-Z0-9\s\(\)\.\-]+)', line, re.IGNORECASE)
    if m:
        key = f"Schedule {m.group(2).split('Schedule')[-1].strip()}"
        key = re.sub(r'\(\)', '', key).strip()
        return key

    # Pattern: Schedule C-All NS items
    m = re.search(r'(Schedule\s*\(?_?\)?\s*[A-Z])-(All\s+(DSR|USSOR|NS)\s+items)', line, re.IGNORECASE)
    if m:
        return f"{m.group(1).strip()}-{m.group(2).strip()}"

    # Pattern: Schedule Schedule B-All USSOR-2021 Items
    m = re.search(r'Schedule\s+(Schedule\s+[A-Z].*Items)', line, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # Pattern: Schedule A-CPWD DSR ITEM 2021 (from BHARUCH tender)
    m = re.search(r'(Schedule\s+[A-Z])-([A-Z\s\d]+)', line, re.IGNORECASE)
    if m:
        return f"{m.group(1).strip()}-{m.group(2).strip()}"

    return None

def category_from_schedule(sched_key):
    """Standardizes schedule names into work categories."""
    sched_upper = str(sched_key).upper()

    if 'EARTHWORK' in sched_upper: return 'EARTHWORK'
    if 'R.C.C' in sched_upper or 'REINFORCED CEMENT CONCRETE' in sched_upper: return 'R.C.C WORK'
    if 'CONCRETE WORK' in sched_upper: return 'CONCRETE WORK'
    if 'STEEL WORK' in sched_upper: return 'STEEL WORK'
    if 'MASONARY' in sched_upper or 'MASONRY' in sched_upper: return 'MASONRY WORK'
    if 'FINISHING' in sched_upper: return 'FINISHING WORK'
    if 'WATER SUPPLY' in sched_upper: return 'WATER SUPPLY'
    if 'WATER PROOFING' in sched_upper: return 'WATER PROOFING'
    if 'DISMANTLING' in sched_upper: return 'DISMANTLING & DEMOLISHING'
    if 'FLOORING' in sched_upper: return 'FLOORING WORK'
    if 'ROOFING' in sched_upper: return 'ROOFING WORK'
    if 'USSOR' in sched_upper: return 'USSOR ITEMS'
    if 'NS' in sched_upper: return 'NS ITEMS' # Broader catch for NS

    m = re.search(r'\(([^)]+)\)', sched_key)
    if m:
        work_type = m.group(1).strip().upper()
        work_type = re.sub(r'^\d+\.\d+\s*', '', work_type)
        if work_type: return work_type

    return sched_key # Fallback

def is_valid_item_row(row_cells):
    """
    Checks if a table row is a valid item by checking the last columns.
    Returns (True, data_dict) or (False, None)
    """
    if len(row_cells) < 5: # Need at least SNo, ItemNo, Desc, Unit, Qty, Rate, Amount (or some combo)
        return False, None

    try:
        # Check from the end: Amount, Rate, Qty
        amount = to_number(row_cells[-1])
        rate = to_number(row_cells[-2])
        qty = to_number(row_cells[-3])
        unit = normalize_text(row_cells[-4])

        if pd.isna(amount) or pd.isna(rate) or pd.isna(qty):
            return False, None

        # Check Unit
        if not re.match(r'^[A-Za-z/%]{1,10}$', unit):
            return False, None

        # CRITICAL VALIDATION: Check math
        if not math.isclose(qty * rate, amount, rel_tol=0.1): # 10% tolerance
            return False, None

        # This is a valid item row. Now find Item No and Description.
        item_no = "N/A"
        s_no = np.nan
        desc_start_index = -1

        # Item No is usually in col 0 or 1
        item_no_col_1 = normalize_text(row_cells[1])
        item_no_col_0 = normalize_text(row_cells[0])

        # Check col 1 first (most common)
        if re.match(r'^[\d\.\-A-Z]+$', item_no_col_1):
            item_no = item_no_col_1
            s_no = to_number(row_cells[0])
            desc_start_index = 2
        # Check col 0 (if S.No is missing)
        elif re.match(r'^[\d\.\-A-Z]+$', item_no_col_0):
            item_no = item_no_col_0
            desc_start_index = 1
        else:
            return False, None # Can't find Item No

        description = " ".join([normalize_text(c) for c in row_cells[desc_start_index:-4]])

        item_data = {
            "S No.": s_no,
            "Item No": item_no,
            "Description of Item": description,
            "Unit": unit,
            "Qty": qty,
            "Rate": rate,
            "Amount": amount
        }
        return True, item_data

    except Exception as e:
        return False, None

# --- Main Parsing Function (v5.1) ---

def parse_tender(pdf_path):

    start_time = time.time()
    progress_logs = []
    confidence_scores = {}

    results = {
        "nit_header": {},
        "schedules_summary_generated": pd.DataFrame(),
        "item_breakups": {},
        "eligibility_criteria": {"bullets": [], "raw_text": ""},
        "flags": [],
        "top10": {},
        "raw_text_pages": [],
        "all_items_df": pd.DataFrame(),
        "num_pages": 0,
        "parser_log": []
    }

    pdf_plumber_pages = []

    # --- Step 1: PDF Text & Table Extraction ---
    try:
        with pdfplumber.open(pdf_path) as pdf:
            results['num_pages'] = len(pdf.pages)
            pdf_plumber_pages = pdf.pages # Store page objects
            progress_logs.append(f"Opened PDF with {results['num_pages']} pages.")

            for i, page in enumerate(pdf_plumber_pages):
                page_no = i + 1
                try:
                    text = page.extract_text(x_tolerance=2, y_tolerance=2) or ""
                    results['raw_text_pages'].append(text)
                except Exception as e:
                    progress_logs.append(f"Warning: Could not extract page {page_no}: {e}")
                    results['raw_text_pages'].append("")

        progress_logs.append(confidence_msg("Text & Tables", "Method 2 (pdfplumber)", 98))
        confidence_scores['text_extraction'] = 98
    except Exception as e:
        progress_logs.append(f"FATAL: Could not open PDF: {e}")
        return results, progress_logs, confidence_scores

    # --- Step 2: NIT Header Parsing (Hybrid) ---
    try:
        page_1_text = results['raw_text_pages'][0]
        page_1_tables = []
        if pdf_plumber_pages:
             page_1_tables = pdf_plumber_pages[0].extract_tables() or []

        results['nit_header'] = parse_nit_header_hybrid(page_1_text, page_1_tables)

        if 'Tender No' not in results['nit_header']:
            raise ValueError("Tender No not found")

        if to_number(results['nit_header'].get('Earnest Money (Rs.)')) is None or to_number(results['nit_header'].get('Earnest Money (Rs.)')) == 0:
            progress_logs.append(confidence_msg("NIT Header", "Hybrid Method", 60))
            confidence_scores['nit_header'] = 60
        else:
            progress_logs.append(confidence_msg("NIT Header", "Hybrid Method", 95))
            confidence_scores['nit_header'] = 95

    except Exception as e:
        progress_logs.append(f"Error parsing NIT Header: {e}")
        confidence_scores['nit_header'] = 30

    # --- Step 3: Hybrid Table-Based Item Parsing (v5.1) ---
    try:
        all_parsed_items = []
        current_schedule_key = "UNKNOWN"

        # Define table settings for pdfplumber's table extraction
        table_settings = {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "snap_tolerance": 3,
            "join_tolerance": 3,
            "min_words_vertical": 2,
        }

        for p_num, page in enumerate(tqdm(pdf_plumber_pages, desc="Parsing Items (Table)", leave=False)):
            page_no = p_num + 1

            # Extract tables using robust settings
            tables = page.extract_tables(table_settings)
            if not tables:
                tables = page.extract_tables() # Fallback to default

            if not tables:
                continue

            for table in tables:
                if not table:
                    continue

                for row in table:
                    # Clean the row: remove None, empty strings
                    row_cells = [str(cell) for cell in row if cell is not None and str(cell).strip()]
                    if not row_cells:
                        continue

                    row_text = " ".join(row_cells)

                    # 1. Check for Schedule Header
                    new_schedule_key = parse_schedule_header(row_text)
                    if new_schedule_key:
                        current_schedule_key = new_schedule_key
                        results['parser_log'].append(f"Page {page_no}: Switched to Schedule '{current_schedule_key}'")
                        continue

                    # 2. Check for Valid Item Row
                    is_item, item_data = is_valid_item_row(row_cells)
                    if is_item:
                        item_data['Page'] = page_no
                        item_data['Schedule'] = current_schedule_key
                        all_parsed_items.append(item_data)
                        continue

                    # 3. Handle Multi-Line Descriptions
                    # If not a header, not an item, and we have a previous item...
                    if all_parsed_items:
                        # ...and the row has few columns (likely just description text)
                        if len(row_cells) < 4 and "Total" not in row_text:
                            # Append this text to the last item's description
                            all_parsed_items[-1]['Description of Item'] += " " + normalize_text(row_text)

        if not all_parsed_items:
            raise ValueError("Table parser found no valid items.")

        # --- Convert to DataFrames ---
        results['all_items_df'] = pd.DataFrame(all_parsed_items)
        results['all_items_df']['Category'] = results['all_items_df']['Schedule'].apply(category_from_schedule)

        for sched_key in results['all_items_df']['Schedule'].unique():
            df = results['all_items_df'][results['all_items_df']['Schedule'] == sched_key].copy()
            cols = ['S No.', 'Item No', 'Description of Item', 'Unit', 'Qty', 'Rate', 'Amount', 'Page', 'Schedule', 'Category']
            for c in cols:
                if c not in df.columns: df[c] = np.nan
            results['item_breakups'][sched_key] = df[cols]

        progress_logs.append(confidence_msg("Item Breakups", "Table-Based (v5.1)", 95))
        confidence_scores['item_breakups'] = 95

    except Exception as e:
        progress_logs.append(f"Item Breakup parsing error: {e}\n{traceback.format_exc()}")
        confidence_scores['item_breakups'] = 30

    # --- Step 4: Generate Schedule Summary (from parsed items) ---
    try:
        if not results['all_items_df'].empty:
            df = results['all_items_df']
            summary = df.groupby('Schedule')['Amount'].agg(['sum', 'count']).reset_index()
            summary.columns = ['Schedule', 'Total Amount', 'Item Count']
            summary = summary.sort_values('Total Amount', ascending=False)
            results['schedules_summary_generated'] = summary
            progress_logs.append(confidence_msg("Schedule Summary", "Generated from Items", 100))
            confidence_scores['schedule_summary'] = 100
        else:
            raise ValueError("all_items_df is empty, cannot generate summary.")
    except Exception as e:
        progress_logs.append(f"Schedule Summary generation error: {e}")
        confidence_scores['schedule_summary'] = 0

    # --- Step 5: Eligibility Criteria ---
    try:
        start_page = 4
        end_page = min(35, results['num_pages'])
        eligibility_text = "\n".join(results['raw_text_pages'][start_page:end_page])

        eligibility_bullets = []
        in_eligibility_section = False

        for line in eligibility_text.split('\n'):
            line_norm = normalize_text(line)
            line_upper = line_norm.upper()

            if 'ELIGIBILITY CONDITIONS' in line_upper or 'FINANCIAL CRITERIA' in line_upper or 'TECHNICAL CRITERIA' in line_upper:
                in_eligibility_section = True

            if not in_eligibility_section:
                continue

            if re.match(r'^\s*(\d+\.\d+|\(i+\)|\([a-z]\)|[a-z]\)|\d+\))\s', line_norm, re.IGNORECASE) or line_norm.startswith('•'):
                if len(line_norm.split()) > 5:
                    eligibility_bullets.append(line_norm)

        results['eligibility_criteria']['bullets'] = eligibility_bullets
        results['eligibility_criteria']['raw_text'] = eligibility_text

        progress_logs.append(confidence_msg("Eligibility Criteria", "Keyword Scan", 80 if eligibility_bullets else 45))
        confidence_scores['eligibility'] = 80 if eligibility_bullets else 45

    except Exception as e:
        progress_logs.append(f"Eligibility parsing error: {e}")
        confidence_scores['eligibility'] = 30

    # --- Step 6: Top 10 Cost Drivers (Hybrid Logic) ---
    try:
        if not results['all_items_df'].empty:
            df = results['all_items_df']
            df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0.0)

            # 1. Process Schedule A
            df_sched_A = df[df['Schedule'].str.contains('Schedule A', case=False, na=False)]
            sched_A_groups = df_sched_A.groupby('Category')['Amount'].sum().reset_index()
            sched_A_groups.rename(columns={'Category': 'Rankable Entity'}, inplace=True)
            sched_A_groups['Type'] = 'Group (Sch A)'

            # 2. Process Other Schedules
            df_other_scheds = df[~df['Schedule'].str.contains('Schedule A', case=False, na=False)]
            other_items = df_other_scheds[['Description of Item', 'Amount']].copy()
            other_items.rename(columns={'Description of Item': 'Rankable Entity'}, inplace=True)
            other_items['Type'] = 'Item'

            # 3. Combine and Rank
            combined_ranking = pd.concat([sched_A_groups, other_items], ignore_index=True)
            combined_ranking = combined_ranking.sort_values('Amount', ascending=False)

            total_amt = combined_ranking['Amount'].sum()
            combined_ranking['Pct'] = (combined_ranking['Amount'] / total_amt * 100) if total_amt > 0 else 0

            results['top10'] = {
                'total_amount': total_amt,
                'drivers': combined_ranking.head(10).to_dict('records')
            }
            progress_logs.append(confidence_msg("Top 10 Drivers", "Hybrid Aggregation", 95))
            confidence_scores['top10'] = 95
        else:
            raise ValueError("all_items_df is empty, cannot calculate Top 10.")

    except Exception as e:
        progress_logs.append(f"Top 10 Drivers error: {e}")
        confidence_scores['top10'] = 20

    # --- Step 7: Flags ---
    try:
        header_flags = results['nit_header']
        if 'No' in header_flags.get('Are JV allowed to bid', 'Yes').upper():
            results['flags'].append("JV NOT ALLOWED")

        if 'Single Packet' in header_flags.get('Bidding System', ''):
            results['flags'].append("Single Packet System")

        em = to_number(header_flags.get('Earnest Money (Rs.)'))
        if em is not None and em > 0:
            results['flags'].append(f"Earnest Money: ₹{em/1_00_000:.2f} Lakh")

        progress_logs.append("✓ Flags identified.")
    except Exception as e:
        progress_logs.append(f"Flags error: {e}")

    end_time = time.time()
    results['parse_time_seconds'] = end_time - start_time
    progress_logs.append(f"Total processing time: {results['parse_time_seconds']:.2f} seconds.")

    return results, progress_logs, confidence_scores

# ----------------------------------------------
# STEP 3: Upload, Execute Parser, and Display Results
# ----------------------------------------------

# --- 3a. Upload File ---
print("Please upload your tender PDF:")
uploaded = files.upload()
pdf_filename = None

if uploaded:
    pdf_filename = next(iter(uploaded))
    print(f"\n✓ Uploaded '{pdf_filename}'")

    # --- 3b. Run the Parser ---
    print(f"--- Analyzing '{pdf_filename}' ---")
    parsed_data, logs, confidences = parse_tender(pdf_filename)

    # --- 3c. Format the Output ---

    # Header
    tender_no = parsed_data['nit_header'].get('Tender No', 'N/A')
    try:
        adv_val_str = f"₹{to_number(parsed_data['nit_header'].get('Advertised Value')) / 1_00_00_000:.2f} Cr"
    except:
        adv_val_str = f"₹{parsed_data['nit_header'].get('Advertised Value', 'N/A')}"

    name_of_work = parsed_data['nit_header'].get('Name of Work', 'N/A')

    md = f"# TENDER {tender_no} | {name_of_work} | {adv_val_str}\n\n"

    # Flags
    if parsed_data['flags']:
        flags_html = " ".join([f"<span style='color:red; font-weight:bold; border: 1px solid red; padding: 2px 5px; border-radius: 5px; margin-right: 10px;'>{flag}</span>" for flag in parsed_data['flags']])
        md += f"{flags_html}\n\n"

    # NIT Header Table
    md += "## 📄 NIT HEADER\n\n"
    if parsed_data['nit_header']:
        header_df = pd.DataFrame(list(parsed_data['nit_header'].items()), columns=['Field', 'Value'])
        md += header_df.to_markdown(index=False) + "\n\n"
    else:
        md += "❌ Could not extract NIT header data.\n\n"

    # Schedule Summary (GENERATED)
    md += "## SCHEDULE SUMMARY (Generated from Item Breakup)\n\n"
    if not parsed_data['schedules_summary_generated'].empty:
        summary_df = parsed_data['schedules_summary_generated'].copy()
        summary_df['Total Amount'] = summary_df['Total Amount'].apply(lambda x: f"₹ {x:,.2f}")
        md += summary_df.to_markdown(index=False) + "\n\n"
    else:
        md += "❌ No item data found, could not generate schedule summary.\n\n"

    # Top 10 Drivers
    md += "## 🎯 TOP 10 COST DRIVERS (Global)\n\n"
    if parsed_data['top10'] and parsed_data['top10'].get('drivers'):
        total = parsed_data['top10']['total_amount']
        md += f"**Total Estimated Value (from parsed items)**: ₹ {total:,.2f}\n\n"

        top10_df = pd.DataFrame(parsed_data['top10']['drivers'])
        # Format the numbers
        top10_df['Amount'] = top10_df['Amount'].apply(lambda x: f"₹ {x:,.2f}")
        top10_df['Pct'] = top10_df['Pct'].apply(lambda x: f"{x:.1f}%")

        # *** START: ERROR FIX ***
        # Rename columns *before* selecting them
        top10_df.rename(columns={
            'Amount': 'Total Amount',
            'Pct': 'Percentage'
        }, inplace=True)
        # *** END: ERROR FIX ***

        # Now this selection will work
        top10_df = top10_df[['Rankable Entity', 'Total Amount', 'Percentage', 'Type']]

        md += top10_df.to_markdown(index=False) + "\n\n"
    else:
        md += "❌ Could not compute cost drivers. Item parsing may have failed.\n\n"

    # Eligibility Criteria
    md += "## ✅ ELIGIBILITY CRITERIA\n\n"
    if parsed_data['eligibility_criteria']['bullets']:
        md += "*(Top 10 extracted criteria)*\n"
        for i, bullet in enumerate(parsed_data['eligibility_criteria']['bullets'][:10]):
            md += f"• {bullet}\n"
    else:
        md += "❌ No specific eligibility criteria bullets found in text scan.\n\n"

    # Full Item Breakup
    md += "## 📊 FULL ITEM BREAKUP (Sample)\n\n"
    if parsed_data['item_breakups']:
        md += f"Found {len(parsed_data['item_breakups'])} schedules and {len(parsed_data['all_items_df'])} total items.\n\n"

        # Sort schedules by total amount
        sorted_schedules = sorted(
            parsed_data['item_breakups'].items(),
            key=lambda item: item[1]['Amount'].sum(),
            reverse=True
        )

        for sched_key, df in sorted_schedules:
            md += f"### {sched_key}\n"
            md += f"**Total: ₹{df['Amount'].sum():,.2f}** ({len(df)} items)\n\n"

            # Display sample (first 5)
            sample_df = df.head(5).copy()
            sample_df['Amount'] = sample_df['Amount'].apply(lambda x: f"{x:,.2f}")
            sample_df['Rate'] = sample_df['Rate'].apply(lambda x: f"{x:,.2f}")

            md += sample_df[['S No.', 'Item No', 'Description of Item', 'Unit', 'Qty', 'Rate', 'Amount']].to_markdown(index=False) + "\n"
            if len(df) > 5:
                md += f"*...and {len(df) - 5} more items.*\n"
            md += "\n---\n"

    else:
        md += "❌ No item breakups extracted.\n\n"

    # --- Display Output ---
    display(Markdown(md))

    # --- Display Logs ---
    print("\n" + "="*60)
    print("📋 PARSING LOGS")
    print("="*60)
    for log in logs:
        print(log)
    if parsed_data['parser_log']:
        print("\n--- Item Parser Log (Sample) ---")
        for log in parsed_data['parser_log'][:10]: # Print first 10
            print(log)
        if len(parsed_data['parser_log']) > 10:
            print(f"...and {len(parsed_data['parser_log']) - 10} more log entries.")

    # --- Export Buttons ---
    print("\n" + "="*60)
    print("📦 EXPORT OPTIONS")
    print("="*60)

    try:
        # 1. Schedule Amounts Summary
        if not parsed_data['schedules_summary_generated'].empty:
            parsed_data['schedules_summary_generated'].to_csv('schedule_amounts.csv', index=False)
            files.download('schedule_amounts.csv')
            print("✓ Downloaded: schedule_amounts.csv")
    except Exception as e:
        print(f"✗ Export error (schedule_amounts): {e}")

    try:
        # 2. Full Breakup
        if not parsed_data['all_items_df'].empty:
            with pd.ExcelWriter('full_breakup.xlsx') as writer:
                # Write all items to one sheet
                parsed_data['all_items_df'].to_excel(writer, sheet_name='All_Items', index=False)
                # Write each schedule to its own sheet
                for sched_key, df in parsed_data['item_breakups'].items():
                    safe_sheet_name = re.sub(r'[\r\n\t\[\]\*\:\?\/]', '', str(sched_key))[:30] # Clean sheet name
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

            files.download('full_breakup.xlsx')
            print("✓ Downloaded: full_breakup.xlsx (includes all items + individual schedule tabs)")
    except Exception as e:
        print(f"✗ Export error (full_breakup): {e}")

else:
    print("⚠️ No file uploaded. Please run this cell again to upload your tender PDF.")

✓ All libraries installed.
Please upload your tender PDF:


Saving NIT-BCT-24-25-257.pdf to NIT-BCT-24-25-257 (1).pdf

✓ Uploaded 'NIT-BCT-24-25-257 (1).pdf'
--- Analyzing 'NIT-BCT-24-25-257 (1).pdf' ---


# TENDER BCT-24-25-257 | VR-JRS Section:- Providing 3 nos RCC OH tank in lieu of old dilapidated RCC storage tank under the jurisdiction of Sr. DEN/North/MMCT. | ₹4.21 Cr

<span style='color:red; font-weight:bold; border: 1px solid red; padding: 2px 5px; border-radius: 5px; margin-right: 10px;'>Single Packet System</span> <span style='color:red; font-weight:bold; border: 1px solid red; padding: 2px 5px; border-radius: 5px; margin-right: 10px;'>Earnest Money: ₹3.61 Lakh</span>

## 📄 NIT HEADER

| Field                               | Value                                                                                                                                  |
|:------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------|
| Name of Work                        | VR-JRS Section:- Providing 3 nos RCC OH tank in lieu of old dilapidated RCC storage tank under the jurisdiction of Sr. DEN/North/MMCT. |
| Bidding type                        | Normal Tender                                                                                                                          |
| Tender Type                         | Open                                                                                                                                   |
| Bidding System                      | Single Packet System                                                                                                                   |
| Tender Closing Date Time            | 04/02/2025 15:00                                                                                                                       |
| Date Time Of Uploading Tender       | 09/01/2025 10:19                                                                                                                       |
| Pre-Bid Conference Required         | No                                                                                                                                     |
| Pre-Bid Conference Date Time        | Not Applicable                                                                                                                         |
| Advertised Value                    | 42145189.36                                                                                                                            |
| Tendering Section                   | CETR/N/II                                                                                                                              |
| Bidding Style                       | [ Decision at Schedule level ]                                                                                                         |
| Earnest Money (Rs.)                 | 360700.00                                                                                                                              |
| Validity of Offer ( Days)           | 60                                                                                                                                     |
| Tender Doc. Cost (Rs.)              | 0.00                                                                                                                                   |
| Period of Completion                | 18 Months                                                                                                                              |
| Contract Type                       | Works - General                                                                                                                        |
| Contract Category                   | Expenditure                                                                                                                            |
| Bidding Start Date                  | 21/01/2025                                                                                                                             |
| Are JV allowed to bid               | No                                                                                                                                     |
| Number of JV Member Allowed         | 0                                                                                                                                      |
| Are Consortium allowed to bid       | No                                                                                                                                     |
| Number of Consortium Member Allowed | 0                                                                                                                                      |
| Ranking Order For Bids              | Lowest to Highest                                                                                                                      |
| Expenditure Type                    | Capital (Works)                                                                                                                        |
| Tender No                           | BCT-24-25-257                                                                                                                          |

## SCHEDULE SUMMARY (Generated from Item Breakup)

| Schedule                        | Total Amount    |   Item Count |
|:--------------------------------|:----------------|-------------:|
| Schedule A-All DSR 2021 Items   | ₹ 53,657,253.98 |           87 |
| Schedule C-All NS items         | ₹ 889,099.20    |            3 |
| Schedule B-All USSOR-2021 Items | ₹ 554,956.65    |            2 |

## 🎯 TOP 10 COST DRIVERS (Global)

**Total Estimated Value (from parsed items)**: ₹ 55,101,309.83

| Rankable Entity                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  | Total Amount    | Percentage   | Type          |
|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------|:-------------|:--------------|
| Schedule A-All DSR 2021 Items                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    | ₹ 53,657,253.98 | 97.4%        | Group (Sch A) |
| JCB Backhoe Loaders 3DX Plus or similar with minimum 1.10 cum bucket capacity Item- 1 Hand packed dry rubble soling                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              | ₹ 507,654.00    | 0.9%         | Item          |
| "Uncoursed hand-packed DRY RUBBLE FILLING (in work like rubble filling behind abutments etc.) without any special dressing of stones, complete." Item- 2 Supply and fixing signage board                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         | ₹ 388,800.00    | 0.7%         | Item          |
| Providing SIGNAGE BOARDS of 2mm thick aluminium sheets backing and pasted with retro reflective sheeting of EG (Engineering Grade) on the background with colour, pattern & Designs as per approved drawings and with Signages cut out of retro reflective sheeting of EG (Engineering grade) of approved pattern, design, text as per approved drawings and super imposed on the first layer. The sizes and shapes of aluminum backing will be as directed by the Railway Engineer and payment will be made for the area of aluminium backing after cutting to required shapes and no extra payment will be made for wastages as well as reflective sheetings separately. The rate shall however exclude the cost of back support mild steel frames for aluminum sheets and vertical posts which will be paid for separately. The cost of excavation. foundation concrete and filling back earth in foundations for the vertical posts will be paid separately. Item- 3 Providing and laying 35 mm thick heavy duty chequered tiles                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             | ₹ 309,247.20    | 0.6%         | Item          |
| "Providing & laying 35mm thick heavy duty chequered tiles like 'Rock tiles' of super group of companies or similar having certification of Bureau of Indian standards. The tiles shall conform to IS 13801-1993 and shall be laid on 20mm bed of Cement mortar 1:3 and joints finished with neat white cement mixed with pigments of matching shade. The tiles shall have a wearing course of 10 to 15mm made of white cement, emery, quartz chips & pigments. The base shall be 20 to 25mm thick of grey cement, rock chips & good quality sand. The colour, shape, size & pattern of tiles shall be got approved by the Railway Engineer or his representative before use on work Sample of tiles shall be got tested for abrasion, water absorption, wet transverse & dry transverse test from approved labs before use on work at contractor's cost. Railway reserves right to test tile samples any time during the progress of work & reject unsuitable lots. The tests shall be as per IS13801-1993. The rate is inclusive of all materials (including white cement but excluding grey cement for fixing the tiles), labour, lead, lift, transportation, taxes etc.. complete." Item- 4 Supply & Fixing brass name plate S.No. Description 1 I/we the tenderer (s) am/are signing this document after carefully reading the contents. 2 I/We the tenderer(s) also accept all the conditions of the tender and have signed all the pages in confirmation thereof. 3 I/we hereby declare that I/we have downloaded the tender documents from Indian Railway website www.ireps.gov.in . I/we have verified the content of the document from the website and there is no addition, no deletion or no alteration to the content of the tender document. In case of any discrepancy noticed at any stage i.e. evaluation of tenders, execution of work or final payment of the contract, the master copy available with the railway Administration shall be final and binding upon me/us. 4 I/we declare and certify that I/we have not made any misleading or false representation in the forms, statements and attachments in proof of the qualification requirements. 5 I/We also understand that my/our offer will be evaluated based on the documents/credentials submitted along with the offer and same shall be binding upon me/us. 6 I/We declare that the information and documents submitted along with the tender by me/us are correct and I/we are fully responsible for the correctness of the information and documents, submitted by us. 7 I/we certify that I/we the tenderer(s) is/are not blacklisted or debarred by Railways or any other Ministry / Department of Govt. of India from participation in tender on the date of submission of bids, either in individual capacity or as a HUF/ member of the partnership firm/LLP/JV/Society/Trust. 8 I/we understand that if the contents of the certificate submitted by us are found to be forged/false at any time during process for evaluation of tenders, it shall lead to forfeiture of the Bid Security and may also lead to any other action provided in the contract including banning of business for a period of upto two year. Further, I/we and all my/our constituents understand that my/our offer shall be summarily rejected. 9 I/we also understand that if the contents of the certificate submitted by us are found to be false/forged at any time after the award of the contract, it will lead to termination of the contract, along with forfeiture of Bid Security/Security Deposit and Performance guarantee and may also lead to any other action provided in the contract including banning of business for a period of upto two year. 10 I/We have read the clause regarding restriction on procurement from a bidder of a country which shares a land border with India and certify that I am/We are not from such a country or, if from such a country, have been registered with the competent Authority. I/We hereby certify that I/we fulfil all the requirements in this regard and am/are eligible to be considered (evidence of valid registration by the competent authority is enclosed) S.No. Description 1 Partnership firm/Joint Venture (JV) / Hindu Undivided Family (HUF) / Limited Liability Partnership (LLP) etc. Please submit a certificate in the prescribed format (please download the format from the link given below). Non submission of the certificate, or submission of certificate either not properly filled in, or in a format other than the prescribed format shall lead to summary rejection of your offer. ( Click here to download the Format of Self Certification) S.No. Document Name Document Description 1 SPLCONDTECHOHTANKDRD.pdf SPECIAL CONDITION TECHINICAL 2 ACS-2toGCC-2022_2022-CE-1-CT-GCC-2022- POLICY_13.12.2022_1.pdf GCC correction slip No. 2 dtd.13.12.2022 3 2022-CE-I-CT- GCCCorrespondencedated.14.05.2024.pdf Clarification regarding submission of Annexure-V 4 BIDCAPACITY_2.pdf BID CAPACITY 20CR ANNEXURE-VI AS PER CORRECTION SLIP 5 1-Safetyrules.pdf Saftey Rule 6 2-guidelineforelectricalcondition.pdf Guideline for electrial connection 7 3-JPOSTandElectfordiggingwork.pdf JPO for cable digging 8 NOrelativecertificate.pdf No relative certificate 9 ProcedureforpaymentofContractorbillasperGST.pdf Procedure of payment of Contractor bill post GST 10 Letterofcreditasmodeofpayment_1.pdf Letter of Credit 11 PCEletterEMDPGSD.pdf PCE letter EMD PG SD 12 2024-CE-I-CAOCWorkshop-part-2.pdf JPO for digging work close to Rly. signalling etc 13 ACS-4.pdf correction slip ACS-4 GCC 14 NoRetiredRailwayEmployeeGCCApril2022.pdf NO RETIRED RAILWAY EMPLOYEE GCC 2022 15 PerformanceSecuritydtd.30.12.2021.pdf Performance Security Rly Board Letter 16 ACS-3.pdf correction slip ACS-3 GCC 17 GCC_April-2022ACS14.07.2022.pdf GCC April 2022 with correction slip 14.07.2022 18 Annexure-VIBAnnualContractualTurnover.pdf ANNEXURE VIB ANNUAL CONTRACTUAL TURNOVER GCC 2022 19 PBGProformaGCC2022.pdf PBG PROFORMA 20 GCC_April-2022ACS14.07.2022-PVCClause.pdf GCC April 2022 ACS 14.07.22 PVC Clause 21 EXEMPTIONOFESICEPFO.pdf EXemption of ESIC 22 SPECIALCONDITIONSCHINAMOSAIC.pdf Special condition China Mosaic 23 Specialconditionofdesignaanddrawing.pdf Special condition of drawing | ₹ 191,052.00    | 0.3%         | Item          |
| Ordinary Portland Cement 53 grade of approved brands/makes Hiring of machinery for minor miscellaneous works for short duration including operator/driver, fuel, lubricants and consumable. The contractor shall arrange all statutory permits as required by rules and regulations prevailing in the area of work. Payment shall be made for actual working hours at site.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      | ₹ 47,302.65     | 0.1%         | Item          |

## ✅ ELIGIBILITY CRITERIA

*(Top 10 extracted criteria)*
• a) The tenderer must have successfully completed or substantially
• 10.1 above, shall be satisfied by either the 'JV in its own name & style' or (Mandatory)
• (b) (1) In case of tenders for composite works (e.g. works involving more
• 1.1 work as per para 10.1 above, shall be satisfied by either the 'JV in its own No No Not Allowed
• (b) (3) To evaluate the technical eligibility of tenderer, only components
• (ii) Existing commitments and balance amount of ongoing works with
• 1.5 has been issued by a person authorized by the Public listed company to No No
• 1.6 Defination of Similar Work :- Construction of RCC overhead tank. No No Not Allowed
• 1) The tenderer shall be required to submit the Bid Security with
• 2.1 Bid Security as mentioned in tender documents, failing which Yes No Not Allowed
## 📊 FULL ITEM BREAKUP (Sample)

Found 3 schedules and 92 total items.

### Schedule A-All DSR 2021 Items
**Total: ₹53,657,253.98** (87 items)

|   S No. | Item No   | Description of Item                                                                                                                                                                                                                                                                                                                                             | Unit   |   Qty | Rate     | Amount     |
|--------:|:----------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------|------:|:---------|:-----------|
|       1 | 2.6.1     | All kinds of soil                                                                                                                                                                                                                                                                                                                                               | cum    |  1600 | 205.45   | 328,720.00 |
|       2 | 2.25      | Filling available excavated earth (excluding rock) in trenches, plinth, sides of foundations etc. in layers not exceeding 20cm in depth, consolidating each deposited layer by ramming and watering, lead up to 50 m and lift upto 1.5 m. 2.26 Extra for every additional lift of 1.5 m or part thereof in excavation / banking excavated or stacked materials. | cum    |   800 | 253.95   | 203,160.00 |
|       3 | 2.26.2    | Ordinary or hard rock                                                                                                                                                                                                                                                                                                                                           | cum    |    80 | 187.40   | 14,992.00  |
|       4 | 2.27      | Supplying and filling in plinth with sand under floors, including watering, ramming, consolidating and dressing complete.                                                                                                                                                                                                                                       | cum    |    80 | 2,161.20 | 172,896.00 |
|       5 | 2.32      | Clearing grass and removal of the rubbish up to a distance of 50 m outside the periphery of the area cleared. 2.33 Felling trees of the girth (measured at a height of 1 m above ground level), including cutting of trunks and branches, removing the roots and stacking of serviceable material and disposal of unserviceable material.                       | Sqm    |   240 | 7.40     | 1,776.00   |
*...and 82 more items.*

---
### Schedule C-All NS items
**Total: ₹889,099.20** (3 items)

|   S No. |   Item No | Description of Item                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              | Unit   |   Qty | Rate     | Amount     |
|--------:|----------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------|------:|:---------|:-----------|
|       1 |         1 | "Uncoursed hand-packed DRY RUBBLE FILLING (in work like rubble filling behind abutments etc.) without any special dressing of stones, complete." Item- 2 Supply and fixing signage board                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         | cum    |   600 | 648.00   | 388,800.00 |
|       1 |         2 | Providing SIGNAGE BOARDS of 2mm thick aluminium sheets backing and pasted with retro reflective sheeting of EG (Engineering Grade) on the background with colour, pattern & Designs as per approved drawings and with Signages cut out of retro reflective sheeting of EG (Engineering grade) of approved pattern, design, text as per approved drawings and super imposed on the first layer. The sizes and shapes of aluminum backing will be as directed by the Railway Engineer and payment will be made for the area of aluminium backing after cutting to required shapes and no extra payment will be made for wastages as well as reflective sheetings separately. The rate shall however exclude the cost of back support mild steel frames for aluminum sheets and vertical posts which will be paid for separately. The cost of excavation. foundation concrete and filling back earth in foundations for the vertical posts will be paid separately. Item- 3 Providing and laying 35 mm thick heavy duty chequered tiles                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             | Sqm    |    60 | 5,154.12 | 309,247.20 |
|       1 |         3 | "Providing & laying 35mm thick heavy duty chequered tiles like 'Rock tiles' of super group of companies or similar having certification of Bureau of Indian standards. The tiles shall conform to IS 13801-1993 and shall be laid on 20mm bed of Cement mortar 1:3 and joints finished with neat white cement mixed with pigments of matching shade. The tiles shall have a wearing course of 10 to 15mm made of white cement, emery, quartz chips & pigments. The base shall be 20 to 25mm thick of grey cement, rock chips & good quality sand. The colour, shape, size & pattern of tiles shall be got approved by the Railway Engineer or his representative before use on work Sample of tiles shall be got tested for abrasion, water absorption, wet transverse & dry transverse test from approved labs before use on work at contractor's cost. Railway reserves right to test tile samples any time during the progress of work & reject unsuitable lots. The tests shall be as per IS13801-1993. The rate is inclusive of all materials (including white cement but excluding grey cement for fixing the tiles), labour, lead, lift, transportation, taxes etc.. complete." Item- 4 Supply & Fixing brass name plate S.No. Description 1 I/we the tenderer (s) am/are signing this document after carefully reading the contents. 2 I/We the tenderer(s) also accept all the conditions of the tender and have signed all the pages in confirmation thereof. 3 I/we hereby declare that I/we have downloaded the tender documents from Indian Railway website www.ireps.gov.in . I/we have verified the content of the document from the website and there is no addition, no deletion or no alteration to the content of the tender document. In case of any discrepancy noticed at any stage i.e. evaluation of tenders, execution of work or final payment of the contract, the master copy available with the railway Administration shall be final and binding upon me/us. 4 I/we declare and certify that I/we have not made any misleading or false representation in the forms, statements and attachments in proof of the qualification requirements. 5 I/We also understand that my/our offer will be evaluated based on the documents/credentials submitted along with the offer and same shall be binding upon me/us. 6 I/We declare that the information and documents submitted along with the tender by me/us are correct and I/we are fully responsible for the correctness of the information and documents, submitted by us. 7 I/we certify that I/we the tenderer(s) is/are not blacklisted or debarred by Railways or any other Ministry / Department of Govt. of India from participation in tender on the date of submission of bids, either in individual capacity or as a HUF/ member of the partnership firm/LLP/JV/Society/Trust. 8 I/we understand that if the contents of the certificate submitted by us are found to be forged/false at any time during process for evaluation of tenders, it shall lead to forfeiture of the Bid Security and may also lead to any other action provided in the contract including banning of business for a period of upto two year. Further, I/we and all my/our constituents understand that my/our offer shall be summarily rejected. 9 I/we also understand that if the contents of the certificate submitted by us are found to be false/forged at any time after the award of the contract, it will lead to termination of the contract, along with forfeiture of Bid Security/Security Deposit and Performance guarantee and may also lead to any other action provided in the contract including banning of business for a period of upto two year. 10 I/We have read the clause regarding restriction on procurement from a bidder of a country which shares a land border with India and certify that I am/We are not from such a country or, if from such a country, have been registered with the competent Authority. I/We hereby certify that I/we fulfil all the requirements in this regard and am/are eligible to be considered (evidence of valid registration by the competent authority is enclosed) S.No. Description 1 Partnership firm/Joint Venture (JV) / Hindu Undivided Family (HUF) / Limited Liability Partnership (LLP) etc. Please submit a certificate in the prescribed format (please download the format from the link given below). Non submission of the certificate, or submission of certificate either not properly filled in, or in a format other than the prescribed format shall lead to summary rejection of your offer. ( Click here to download the Format of Self Certification) S.No. Document Name Document Description 1 SPLCONDTECHOHTANKDRD.pdf SPECIAL CONDITION TECHINICAL 2 ACS-2toGCC-2022_2022-CE-1-CT-GCC-2022- POLICY_13.12.2022_1.pdf GCC correction slip No. 2 dtd.13.12.2022 3 2022-CE-I-CT- GCCCorrespondencedated.14.05.2024.pdf Clarification regarding submission of Annexure-V 4 BIDCAPACITY_2.pdf BID CAPACITY 20CR ANNEXURE-VI AS PER CORRECTION SLIP 5 1-Safetyrules.pdf Saftey Rule 6 2-guidelineforelectricalcondition.pdf Guideline for electrial connection 7 3-JPOSTandElectfordiggingwork.pdf JPO for cable digging 8 NOrelativecertificate.pdf No relative certificate 9 ProcedureforpaymentofContractorbillasperGST.pdf Procedure of payment of Contractor bill post GST 10 Letterofcreditasmodeofpayment_1.pdf Letter of Credit 11 PCEletterEMDPGSD.pdf PCE letter EMD PG SD 12 2024-CE-I-CAOCWorkshop-part-2.pdf JPO for digging work close to Rly. signalling etc 13 ACS-4.pdf correction slip ACS-4 GCC 14 NoRetiredRailwayEmployeeGCCApril2022.pdf NO RETIRED RAILWAY EMPLOYEE GCC 2022 15 PerformanceSecuritydtd.30.12.2021.pdf Performance Security Rly Board Letter 16 ACS-3.pdf correction slip ACS-3 GCC 17 GCC_April-2022ACS14.07.2022.pdf GCC April 2022 with correction slip 14.07.2022 18 Annexure-VIBAnnualContractualTurnover.pdf ANNEXURE VIB ANNUAL CONTRACTUAL TURNOVER GCC 2022 19 PBGProformaGCC2022.pdf PBG PROFORMA 20 GCC_April-2022ACS14.07.2022-PVCClause.pdf GCC April 2022 ACS 14.07.22 PVC Clause 21 EXEMPTIONOFESICEPFO.pdf EXemption of ESIC 22 SPECIALCONDITIONSCHINAMOSAIC.pdf Special condition China Mosaic 23 Specialconditionofdesignaanddrawing.pdf Special condition of drawing | Sqm    |   366 | 522.00   | 191,052.00 |

---
### Schedule B-All USSOR-2021 Items
**Total: ₹554,956.65** (2 items)

|   S No. |   Item No | Description of Item                                                                                                                                                                                                                                                                                                                                                         | Unit   |   Qty | Rate     | Amount     |
|--------:|----------:|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------|------:|:---------|:-----------|
|       1 |    025072 | Ordinary Portland Cement 53 grade of approved brands/makes Hiring of machinery for minor miscellaneous works for short duration including operator/driver, fuel, lubricants and consumable. The contractor shall arrange all statutory permits as required by rules and regulations prevailing in the area of work. Payment shall be made for actual working hours at site. | MT     |   5.1 | 9,275.03 | 47,302.65  |
|       2 |    211201 | JCB Backhoe Loaders 3DX Plus or similar with minimum 1.10 cum bucket capacity Item- 1 Hand packed dry rubble soling                                                                                                                                                                                                                                                         | Hour   | 600   | 846.09   | 507,654.00 |

---



📋 PARSING LOGS
Opened PDF with 33 pages.
✅ Text & Tables → Method 2 (pdfplumber) | 98% confidence
✅ NIT Header → Hybrid Method | 95% confidence
✅ Item Breakups → Table-Based (v5.1) | 95% confidence
✅ Schedule Summary → Generated from Items | 100% confidence
⚠️ Eligibility Criteria → Keyword Scan | 80% confidence
✅ Top 10 Drivers → Hybrid Aggregation | 95% confidence
✓ Flags identified.
Total processing time: 16.91 seconds.

--- Item Parser Log (Sample) ---
Page 2: Switched to Schedule 'Schedule () C-All NS items'
Page 2: Switched to Schedule 'Schedule A-All DSR 2021 Items'
Page 14: Switched to Schedule 'Schedule B-All USSOR-2021 Items'
Page 15: Switched to Schedule 'Schedule C-All NS items'

📦 EXPORT OPTIONS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: schedule_amounts.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: full_breakup.xlsx (includes all items + individual schedule tabs)
